# Task 13 - Product Tag Co-occurrence Analysis

**Goal:** discover product-catalog tags that commonly appear together and summarize tag patterns.

**Input:** `data/processed/womens_clothing_final.csv`
**Outputs:** product-tag features in `data/processed/` and reports in `reports/`.

> The available data describes product listings and has no customer, order, or transaction ID. Each listing is treated as one set of tags; results describe catalog co-occurrence, not items bought together.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project folder containing data/processed.")

ROOT = find_project_root()
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROC / "womens_clothing_final.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected product catalog CSV at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
print(f"Project: {ROOT}\nLoaded: {DATA_PATH.name} | rows={len(df):,} | columns={len(df.columns)}")

def require_columns(frame, columns):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{DATA_PATH.name} is not the expected product-catalog dataset. "
                       f"Missing columns: {missing}. Available columns: {list(frame.columns)}")

from mlxtend.frequent_patterns import apriori, association_rules

require_columns(df, ["title", "rating", "units_sold", "tags"])
products = df.reset_index(drop=True).copy()
products["listing_id"] = np.arange(len(products))
products["tag_list"] = products["tags"].fillna("").astype(str).str.split(",")
products["tag_list"] = products["tag_list"].map(
    lambda values: list(dict.fromkeys(value.strip().casefold() for value in values if value.strip()))
)
tag_rows = products[["listing_id", "tag_list"]].explode("tag_list").dropna(subset=["tag_list"])
tag_rows = tag_rows.rename(columns={"tag_list": "tag"})
tag_counts = tag_rows["tag"].value_counts()
selected_tags = tag_counts.head(40).index
tag_rows = tag_rows[tag_rows["tag"].isin(selected_tags)]
onehot = pd.crosstab(tag_rows["listing_id"], tag_rows["tag"]).reindex(
    products["listing_id"], fill_value=0
).gt(0)
print(f"Product listings: {len(products):,} | unique tags: {len(tag_counts):,} | "
      f"tags used for rules: {len(selected_tags)}")

Project: D:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project
Loaded: womens_clothing_final.csv | rows=1,539 | columns=40
Product listings: 1,539 | unique tags: 2,620 | tags used for rules: 40


In [2]:
support_used = 0.02
itemsets = apriori(onehot, min_support=support_used, use_colnames=True, max_len=3)
multi_itemsets = itemsets[itemsets["itemsets"].map(len) >= 2]
if multi_itemsets.empty:
    raise ValueError(f"No multi-tag itemsets found at support={support_used}. "
                     "Lower support_used after inspecting the tag counts.")
try:
    rules = association_rules(itemsets, metric="confidence", min_threshold=0.1)
except TypeError:
    rules = association_rules(itemsets, num_itemsets=len(onehot),
                              metric="confidence", min_threshold=0.1)
rules = rules.loc[rules["lift"] > 1].copy()
rules["antecedents"] = rules["antecedents"].map(lambda values: ", ".join(sorted(values)))
rules["consequents"] = rules["consequents"].map(lambda values: ", ".join(sorted(values)))
rule_columns = ["antecedents", "consequents", "antecedent support", "consequent support",
                "support", "confidence", "lift", "leverage", "conviction"]
rules = rules[[column for column in rule_columns if column in rules.columns]]
rules = rules.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)

tag_frequency = tag_counts.rename_axis("tag").reset_index(name="product_count")
tag_frequency["product_share"] = tag_frequency["product_count"] / len(products)
tag_support = tag_counts / len(products)
features = products.drop(columns=["listing_id", "tag_list"]).copy()
features["catalog_tag_count"] = products["tag_list"].str.len()
features["product_top_tag"] = products["tag_list"].str[0]
features["product_mean_tag_support"] = products["tag_list"].map(
    lambda values: float(tag_support.reindex(values).mean()) if values else 0.0
)
single_rules = rules.loc[
    ~rules["antecedents"].str.contains(",", regex=False)
    & ~rules["consequents"].str.contains(",", regex=False)
]
best_rules = single_rules.drop_duplicates("antecedents").set_index("antecedents")
features["top_associated_tag"] = features["product_top_tag"].map(best_rules["consequents"])
features["top_association_lift"] = features["product_top_tag"].map(best_rules["lift"])

features.to_csv(PROC / "womens_clothing_product_tag_features.csv", index=False)
rules.to_csv(REPORTS / "task_13_product_tag_rules.csv", index=False)
tag_frequency.to_csv(REPORTS / "task_13_product_tag_frequency.csv", index=False)
status = pd.DataFrame([{
    "task": "Task 13 - Product Tag Co-occurrence Analysis", "status": "completed",
    "run_at": datetime.now().isoformat(timespec="seconds"), "input_file": DATA_PATH.name,
    "rows_in": len(df), "rows_out": len(features), "n_tags_analyzed": len(selected_tags),
    "min_support": support_used, "n_rules": len(rules),
    "output_dataset": "womens_clothing_product_tag_features.csv",
}])
status.to_csv(REPORTS / "task_13_analysis_status.csv", index=False)
display(tag_frequency.head(20), rules.head(15), status.T)

,tag,product_count,product_share
0,women's fashion,1288,0.836907
1,summer,1287,0.836257
2,fashion,1061,0.689409
3,women,947,0.615335
4,casual,882,0.573099
5,plus size,623,0.404808
6,sleeveless,563,0.365822
7,dress,536,0.348278
8,shorts,527,0.342430
9,tops,496,0.322287


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,"mini, sleeveless dress",mini dress,0.022742,0.107212,0.022742,1.000000,9.327273,0.020304,inf
1,mini dress,"mini, sleeveless dress",0.107212,0.022742,0.022742,0.212121,9.327273,0.020304,1.240366
2,"floral, mini dress",mini,0.021442,0.102014,0.020143,0.939394,9.208454,0.017956,14.816764
3,mini,"floral, mini dress",0.102014,0.021442,0.020143,0.197452,9.208454,0.017956,1.219314
4,"floral, mini",mini dress,0.020793,0.107212,0.020143,0.968750,9.035795,0.017914,28.569201
5,mini dress,"floral, mini",0.107212,0.020793,0.020143,0.187879,9.035795,0.017914,1.205740
6,"mini, print",mini dress,0.021442,0.107212,0.020143,0.939394,8.761983,0.017844,14.730994
7,mini dress,"mini, print",0.107212,0.021442,0.020143,0.187879,8.761983,0.017844,1.204940
8,"mini dress, sexy",mini,0.039636,0.102014,0.035088,0.885246,8.677665,0.031044,7.825304
9,mini,"mini dress, sexy",0.102014,0.039636,0.035088,0.343949,8.677665,0.031044,1.463856


,0
task,Task 13 - Product Tag Co-occurrence Analysis
status,completed
run_at,2026-11-20T15:11:31
input_file,womens_clothing_final.csv
rows_in,1539
rows_out,1539
n_tags_analyzed,40
min_support,0.02
n_rules,11706
output_dataset,womens_clothing_product_tag_features.csv
